# 10 — Locked Categorical Evaluation

This notebook records the first untouched categorical evaluation of the
locked probability model.

The ten-date holdout block and thirty-date June external block are evaluated
separately. The selected model, continuous dispersion scale and uniform
probability-mixing parameter remain unchanged.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


def locate_repository(start: Path) -> Path:
    current = start.resolve()

    for candidate in (current, *current.parents):
        if (
            candidate
            / "data/manifests/"
            "10_categorical_evaluation_manifest.json"
        ).exists():
            return candidate

    raise FileNotFoundError("Repository root not found.")


ROOT = locate_repository(Path.cwd())

manifest = json.loads(
    (
        ROOT
        / "data/manifests/"
        "10_categorical_evaluation_manifest.json"
    ).read_text(encoding="utf-8")
)

summary = pd.read_csv(
    ROOT
    / "outputs/final_tables/"
    "10_locked_categorical_block_summary.csv"
)

scores = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "10_locked_categorical_score_panel.csv"
)

checks = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "10_locked_categorical_integrity_checks.csv"
)

print("Status:", manifest["status"])
print("Selected model:", manifest["selected_model"])
print(
    "Continuous scale:",
    manifest["locked_continuous_dispersion_scale"],
)
print(
    "Uniform mixing parameter:",
    manifest["locked_uniform_mixing_lambda"],
)

Status: LOCKED_CATEGORICAL_EVALUATION_COMPLETE
Selected model: pooled_empirical_residual
Continuous scale: 1.25
Uniform mixing parameter: 0.01


## Scores

For realised event \(J_d\) and probability vector
\(\widehat{\boldsymbol p}_{d,r}\), the categorical log score is

\[
-\log \widehat p_{d,r,J_d}.
\]

The multiclass Brier score is

\[
\sum_{j=1}^{11}
\left(
\widehat p_{d,r,j}
-
\mathbf{1}_{\{J_d=j\}}
\right)^2.
\]

Scores from the four decision rules are first averaged within each settlement
date. Settlement date is therefore the uncertainty unit.

In [2]:
display_columns = [
    "chronology_block",
    "dates",
    "probability_books",
    "raw_zero_probability_books",
    "mean_date_regularised_log_score",
    "standard_error_date_regularised_log_score",
    "mean_date_raw_brier_score",
    "mean_date_regularised_brier_score",
    "mean_date_brier_difference_regularised_minus_raw",
]

print(summary[display_columns].to_string(index=False))

assert set(summary["chronology_block"]) == {
    "holdout",
    "external_test",
}

holdout = summary.loc[
    summary["chronology_block"].eq("holdout")
].iloc[0]

external = summary.loc[
    summary["chronology_block"].eq("external_test")
].iloc[0]

assert int(holdout["dates"]) == 10
assert int(holdout["probability_books"]) == 40
assert int(external["dates"]) == 30
assert int(external["probability_books"]) == 119

assert int(holdout["raw_zero_probability_books"]) == 0
assert int(external["raw_zero_probability_books"]) == 1

chronology_block  dates  probability_books  raw_zero_probability_books  mean_date_regularised_log_score  standard_error_date_regularised_log_score  mean_date_raw_brier_score  mean_date_regularised_brier_score  mean_date_brier_difference_regularised_minus_raw
   external_test     30                119                           1                         1.513592                                   0.098432                   0.718773                           0.719350                                          0.000577
         holdout     10                 40                           0                         1.140714                                   0.146420                   0.583216                           0.584306                                          0.001090


## Effect of probability regularisation

The locked one per cent uniform mixture removes exact zero probabilities.

One June forecast assigned zero raw probability to the realised event.
Consequently, its unregularised categorical log score is infinite. The
regularised log scores are finite for every evaluation forecast.

The corresponding increase in multiclass Brier score is small and is reported
rather than hidden.

In [3]:
assert np.isfinite(
    scores["regularised_categorical_log_score"]
).all()

assert (
    scores.loc[
        scores["chronology_block"].eq("external_test"),
        "raw_log_score_finite",
    ]
    .astype(str)
    .str.lower()
    .isin({"false", "0"})
    .sum()
    == 1
)

assert (
    summary[
        "mean_date_brier_difference_regularised_minus_raw"
    ]
    > 0.0
).all()

assert (
    summary[
        "mean_date_brier_difference_regularised_minus_raw"
    ]
    < 0.01
).all()

print(
    "All regularised categorical log scores are finite:",
    True,
)

All regularised categorical log scores are finite: True


## Evidential boundary

Realised HKO outcomes are used only for locked evaluation.

No model or calibration component is reselected. Market prices and trading
returns remain outside this notebook.

In [4]:
assert manifest["model_reselected"] is False
assert manifest["continuous_calibration_reselected"] is False
assert manifest["probability_calibration_reselected"] is False
assert manifest["market_prices_accessed"] is False
assert manifest["trading_returns_calculated"] is False

passed = (
    checks["passed"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin({"true", "1"})
)

assert passed.all()

print("All Notebook 10 integrity checks passed:", True)

All Notebook 10 integrity checks passed: True
